In [1]:
import pandas as pd

In [15]:
def merge_datasets(csv_path, pcap_path):
    # 1. Wczytanie danych z plików
    df_csv = pd.read_csv(csv_path)
    df_pcap = pd.read_csv(pcap_path)

    # 2. Ujednolicenie czasu SWaT (przeliczenie z GMT+8 na UTC i narzucenie typu datetime64[ns])
    df_csv["datetime"] = (
        pd.to_datetime(df_csv["Timestamp_GMT8"])
        .dt.tz_localize("Asia/Singapore")  # Określenie strefy GMT+8
        .dt.tz_convert("UTC")  # Konwersja do UTC
        .dt.tz_localize(None)  # Usunięcie znacznika strefy (naive datetime)
        .astype("datetime64[ns]")  # Naprawa MergeError: wymuszenie typu [ns]
    )

    # 3. Ujednolicenie czasu PCAP (konwersja sekundowego timestampu Unix na UTC)
    df_pcap["datetime"] = (
        pd.to_datetime(df_pcap["sec_timestamp"], unit="s")
        .astype("datetime64[ns]")  # Naprawa MergeError: wymuszenie typu [ns]
    )

    # 4. Sortowanie po czasie (wymóg konieczny dla merge_asof)
    df_csv = df_csv.sort_values(by="datetime")
    df_pcap = df_pcap.sort_values(by="datetime")

    # 5. Połączenie danych po czasie
    df_merged = pd.merge_asof(
        df_csv,
        df_pcap,
        on="datetime",
        direction="nearest",
        tolerance=pd.Timedelta("2s"),  # Tolerancja dopasowania rekordów (np. 5 sekund)
    )

    # 6. Wypełnienie brakujących wartości zerami
    df_merged.fillna(0, inplace=True)

    return df_merged

In [16]:
df = merge_datasets('data\\SWaT_processed_data.csv', 'data\\pcap_network_features.csv')

In [24]:
def extract_features(df, windows=[5, 10, 30]):
    df_features = df.copy()

    df_features["bytes_per_pkt"] = df_features["total_bytes"] / (df_features["pkt_count"] + 1e-5)
    df_features["pkt_count_diff"] = df_features["pkt_count"].diff().fillna(0)

    target_cols = ["pkt_count", "total_bytes", "LIT101.Pv", "FIT101.Pv"]

    for w in windows:
        for col in target_cols:
            df_features[f"{col}_mean_{w}s"] = (
                df_features[col].rolling(window=w, min_periods=1).mean()
            )
            df_features[f"{col}_std_{w}s"] = (
                df_features[col].rolling(window=w, min_periods=1).std().fillna(0)
            )

    return df_features

In [26]:
df_feat = extract_features(df, windows=[5, 10, 30] )
print(df_feat.head())

   P1_STATE  LIT101.Pv  FIT101.Pv  MV101.Status  P101.Status  P102.Status  \
0  0.882401  -1.085159    -0.3966      -0.25818      0.77455          0.0   
1  0.882401  -1.078864    -0.3966      -0.25818      0.77455          0.0   
2  0.882401  -1.072570    -0.3966      -0.25818      0.77455          0.0   
3  0.882401  -1.064339    -0.3966      -0.25818      0.77455          0.0   
4  0.882401  -1.059012    -0.3966      -0.25818      0.77455          0.0   

   P2_STATE  FIT201.Pv  AIT201.Pv  AIT202.Pv  ...  FIT101.Pv_mean_10s  \
0       0.0   0.763963  -0.536987   0.337019  ...             -0.3966   
1       0.0   0.762480  -0.536987   0.337019  ...             -0.3966   
2       0.0   0.762252  -0.536987   0.344806  ...             -0.3966   
3       0.0   0.761225  -0.536987   0.348700  ...             -0.3966   
4       0.0   0.761225  -0.536987   0.348700  ...             -0.3966   

   FIT101.Pv_std_10s  pkt_count_mean_30s  pkt_count_std_30s  \
0                0.0               

In [27]:
print(df_feat.describe())

           P1_STATE     LIT101.Pv     FIT101.Pv  MV101.Status   P101.Status  \
count  13201.000000  1.320100e+04  1.320100e+04  1.320100e+04  1.320100e+04   
mean       0.000000 -1.894637e-16 -4.305993e-17  3.789274e-16 -2.152997e-16   
min       -1.133272 -3.165287e+00 -3.965995e-01 -3.962787e+00 -1.291073e+00   
25%       -1.133272 -3.322255e-01 -3.965995e-01 -2.581803e-01 -1.291073e+00   
50%        0.882401  3.253199e-01 -3.965995e-01 -2.581803e-01  7.745497e-01   
75%        0.882401  6.812074e-01 -3.965995e-01 -2.581803e-01  7.745497e-01   
max        0.882401  1.071957e+00  3.056567e+00  3.446427e+00  7.745497e-01   
std        1.000038  1.000038e+00  1.000038e+00  1.000038e+00  1.000038e+00   

       P102.Status  P2_STATE     FIT201.Pv     AIT201.Pv     AIT202.Pv  ...  \
count      13201.0   13201.0  1.320100e+04  1.320100e+04  1.320100e+04  ...   
mean           0.0       0.0  4.305993e-17  4.736593e-16 -7.836908e-16  ...   
min            0.0       0.0 -1.295042e+00 -1.45472